# Hospital Daily Summary — Silver to Gold

Builds `adb_caresynce.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, patient/doctor counts, abnormal/critical rates, and day-over-day trend.

**Source tables:** `silver.lab_results`, `silver.hospitals`, `silver.calendar`  
**Load strategy:** Full rebuild (CREATE OR REPLACE TABLE)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresynce.gold

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresynce.gold.hospital_daily_summary AS
WITH base AS (
    SELECT
        lr.hospital_id,
        h.hospital_name,
        lr.test_date                                                       AS summary_date,
        c.day_name,
        c.is_weekend,
        COUNT(*)                                                           AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                      AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                       AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)         AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)         AS critical_test_count
    FROM adb_caresynce.silver.lab_results lr
    LEFT JOIN adb_caresynce.silver.hospitals h
        ON lr.hospital_id = h.hospital_id
    LEFT JOIN adb_caresynce.silver.calendar c
        ON c.date_key = CAST(DATE_FORMAT(lr.test_date, 'yyyyMMdd') AS INT)
    GROUP BY
        lr.hospital_id,
        h.hospital_name,
        lr.test_date,
        c.day_name,
        c.is_weekend
)
SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    CAST(ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2))  AS critical_rate_pct,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)      AS prev_day_total_tests,
    CAST(
        ROUND(
            (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
            * 100.0
            / NULLIF(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date), 0),
            2
        ) AS DECIMAL(6,2)
    )                                                                          AS test_volume_change_pct
FROM base

In [0]:
%sql
SELECT * FROM adb_caresynce.gold.hospital_daily_summary LIMIT 20

In [0]:
updated_df = spark.table("adb_caresynce.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))